In [2]:
#the pip commands to be used here

%pip install requests
%pip install pandas
%pip install seaborn


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path

import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
script_dir = Path().resolve().parent

def create_url(base_url,first_part_endpoint,second_part_endpoint):
    return base_url + first_part_endpoint + second_part_endpoint
def getData(url):
    response = requests.get(url)

    # Check response status
    if response.status_code == 200:
        data = response.json()
        print(f"Received {len(data)} records.")
        return data
    else:
        print(f"Error: {response.status_code}")

def saveDataIntoDataFolder(url,data_file_name):
    data_folder = script_dir / 'data'
    
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (data_file_name + ".json")
    
    data = getData(url)
    
    if data:
        with open(file_path, 'w') as file:
            json.dump(data, file)
        print(f"Data saved to {file_path}")
    else:
        print("No data to save.")

base_url = "http://192.168.28.129:8080/"
first_part_endpoint = "dataAnalysisEndpoints/"
second_part_endpoint = "getAllUserInputsExperimentState"        

url = create_url(base_url,first_part_endpoint,second_part_endpoint)

saveDataIntoDataFolder(url,"UserInputs")

Received 83 records.
Data saved to /home/bob/Documents/Diploma project code/Diploma-Project/data/UserInputs.json


In [5]:


def loadDataFromFile(file_name):
    script_dir = Path().resolve()
    file_path = script_dir / 'data' / (file_name + '.json')
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None

In [6]:
df_userInputData = loadDataFromFile("UserInputs")

Loaded 81 records from /home/bob/Documents/Diploma project code/Diploma-Project/dataAnalysis and machine learning/data/UserInputs.json


In [7]:
df_userInputData

,_id,experimentState,timestamp,userInputCategory,details
0,{'$oid': '6849ace703a598ff63a65bf8'},StartingExperiment,{'$date': 1749658855857},ExperimentState,NaN
1,{'$oid': '6849aefb03a598ff63a6667f'},InsertingSourcePollutant,{'$date': 1749659387774},ExperimentState,"{'are-doors-opened': 'on', 'are-people-inside'..."
2,{'$oid': '6849b77503a598ff63a68c90'},RemovingSourcePollutant,{'$date': 1749661557509},ExperimentState,NaN
3,{'$oid': '6849ba8d03a598ff63a69cd3'},InsertingSourcePollutant,{'$date': 1749662349567},ExperimentState,"{'are-doors-opened': 'on', 'are-people-inside'..."
4,{'$oid': '6849c2dc03a598ff63a6c299'},RemovingSourcePollutant,{'$date': 1749664476491},ExperimentState,NaN
...,...,...,...,...,...
76,{'$oid': '6850a2581a9c68f509ae488d'},InsertingSourcePollutant,{'$date': 1750114904144},ExperimentState,"{'are-doors-opened': 'on', 'front-wall': '1.7'..."
77,{'$oid': '6850a5391a9c68f509ae4e80'},RemovingSourcePollutant,{'$date': 1750115641436},ExperimentState,NaN
78,{'$oid': '68514b9900ef5c1f25a18498'},StartingExperiment,{'$date': 1750158233769},ExperimentState,NaN
79,{'$oid': '68514d3400ef5c1f25a18726'},InsertingSourcePollutant,{'$date': 1750158644637},ExperimentState,"{'are-doors-opened': 'on', 'front-wall': '2.1'..."


In [8]:
if 'details' in df_userInputData.columns:
    details_df = df_userInputData["details"].apply(pd.Series)
    
    # Then join with the original DataFrame (drop the nested 'details' if desired)
    df_userInputData = pd.concat([df_userInputData.drop(columns=["details"]), details_df], axis=1)

In [9]:
df_userInputData.columns

Index([               '_id',    'experimentState',          'timestamp',
        'userInputCategory',                    0,   'are-doors-opened',
        'are-people-inside', 'are-windows-opened',         'front-wall',
                'item-used',              'notes',     'pollutant-type',
            'quantity-used',               'room',    'side-right-wall'],
      dtype='object')

In [10]:
if 0 in df_userInputData.columns:
    df_userInputData =df_userInputData.drop(columns=[0])

In [11]:
# Convert epoch (assumes seconds) to datetime in local time
df_userInputData["epoch_ms"] = df_userInputData["timestamp"].apply(lambda x: x["$date"])
df_userInputData["timestamp_local"] = pd.to_datetime(df_userInputData["epoch_ms"], unit="ms").dt.tz_localize("UTC").dt.tz_convert("Europe/Athens")
df_userInputData.drop(columns = ["timestamp"])

,_id,experimentState,userInputCategory,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,pollutant-type,quantity-used,room,side-right-wall,epoch_ms,timestamp_local
0,{'$oid': '6849ace703a598ff63a65bf8'},StartingExperiment,ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1749658855857,2025-06-11 19:20:55.857000+03:00
1,{'$oid': '6849aefb03a598ff63a6667f'},InsertingSourcePollutant,ExperimentState,on,on,on,1.5,φαρμακευτική λοτιόν,,VOC,δύο κουταλιές της σούπας,Σαλόνι,1.7,1749659387774,2025-06-11 19:29:47.774000+03:00
2,{'$oid': '6849b77503a598ff63a68c90'},RemovingSourcePollutant,ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1749661557509,2025-06-11 20:05:57.509000+03:00
3,{'$oid': '6849ba8d03a598ff63a69cd3'},InsertingSourcePollutant,ExperimentState,on,on,on,1.5,Φαρμακευτικό αλκοόλ,,VOC,τρεις κουταλιά της σούπας,Σαλόνι,1.7,1749662349567,2025-06-11 20:19:09.567000+03:00
4,{'$oid': '6849c2dc03a598ff63a6c299'},RemovingSourcePollutant,ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1749664476491,2025-06-11 20:54:36.491000+03:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,{'$oid': '6850a2581a9c68f509ae488d'},InsertingSourcePollutant,ExperimentState,on,NaN,NaN,1.7,Φαρμακευτικό αλκοόλ 95%,,VOC,30 ml,Κρεβατοκάμαρα,1.7,1750114904144,2025-06-17 02:01:44.144000+03:00
77,{'$oid': '6850a5391a9c68f509ae4e80'},RemovingSourcePollutant,ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1750115641436,2025-06-17 02:14:01.436000+03:00
78,{'$oid': '68514b9900ef5c1f25a18498'},StartingExperiment,ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1750158233769,2025-06-17 14:03:53.769000+03:00
79,{'$oid': '68514d3400ef5c1f25a18726'},InsertingSourcePollutant,ExperimentState,on,NaN,NaN,2.1,Φαρμακευτικό αλκοόλ 95%,,VOC,30 ml,Κρεβατοκάμαρα,0.4,1750158644637,2025-06-17 14:10:44.637000+03:00


In [12]:
df_userInputData[df_userInputData["item-used"] == "Φαρμακευτικό αλκοόλ 95%"]

,_id,experimentState,timestamp,userInputCategory,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,pollutant-type,quantity-used,room,side-right-wall,epoch_ms,timestamp_local
19,{'$oid': '684afcdf8ad2cd52978f638d'},InsertingSourcePollutant,{'$date': 1749744863788},ExperimentState,on,on,on,1.5,Φαρμακευτικό αλκοόλ 95%,,VOC,μία κουταλιά της σούπας,Σαλόνι,1.5,1749744863788,2025-06-12 19:14:23.788000+03:00
21,{'$oid': '684b12168ad2cd52978f92ea'},InsertingSourcePollutant,{'$date': 1749750294045},ExperimentState,on,on,on,1.5,Φαρμακευτικό αλκοόλ 95%,,VOC,13 ml περιπου,σαλονι,1.5,1749750294045,2025-06-12 20:44:54.045000+03:00
23,{'$oid': '684b21cb8ad2cd52978fdeca'},InsertingSourcePollutant,{'$date': 1749754315351},ExperimentState,on,on,on,1.5,Φαρμακευτικό αλκοόλ 95%,,VOC,17 ml,Σαλόνι,1.7,1749754315351,2025-06-12 21:51:55.351000+03:00
25,{'$oid': '684b3582440322b264da457e'},InsertingSourcePollutant,{'$date': 1749759362051},ExperimentState,on,on,on,1.5,Φαρμακευτικό αλκοόλ 95%,,VOC,10 ml,Σαλόνι,1.7,1749759362051,2025-06-12 23:16:02.051000+03:00
27,{'$oid': '684b4244c3e31ce70438ae27'},InsertingSourcePollutant,{'$date': 1749762628817},ExperimentState,on,on,on,1.5,Φαρμακευτικό αλκοόλ 95%,,VOC,13,Σαλόνι,1.7,1749762628817,2025-06-13 00:10:28.817000+03:00
35,{'$oid': '684c1c2f13c178db70d865be'},InsertingSourcePollutant,{'$date': 1749818415302},ExperimentState,on,NaN,on,1.5,Φαρμακευτικό αλκοόλ 95%,,VOC,15 ml,Σαλόνι,1.7,1749818415302,2025-06-13 15:40:15.302000+03:00
37,{'$oid': '684c47ff13c178db70d93716'},InsertingSourcePollutant,{'$date': 1749829631063},ExperimentState,on,NaN,NaN,0.75,Φαρμακευτικό αλκοόλ 95%,"Είναι στην θέση 39,59",VOC,15 ml,Κρεβατοκάμαρα,1.83,1749829631063,2025-06-13 18:47:11.063000+03:00
39,{'$oid': '684c550e13c178db70d97664'},InsertingSourcePollutant,{'$date': 1749832974427},ExperimentState,on,NaN,NaN,0.75,Φαρμακευτικό αλκοόλ 95%,,VOC,15 ml,Κρεβατοκάμαρα,1.83,1749832974427,2025-06-13 19:42:54.427000+03:00
41,{'$oid': '684c609813c178db70d9b161'},InsertingSourcePollutant,{'$date': 1749835928190},ExperimentState,on,NaN,NaN,0.75,Φαρμακευτικό αλκοόλ 95%,,VOC,15 ml,Κρεβατοκάμαρα,1.83,1749835928190,2025-06-13 20:32:08.190000+03:00
43,{'$oid': '684c7f2c13c178db70da3bea'},InsertingSourcePollutant,{'$date': 1749843756194},ExperimentState,NaN,NaN,NaN,0.75,Φαρμακευτικό αλκοόλ 95%,,VOC,15 ml,Κρεβατοκάμαρα,1.83,1749843756194,2025-06-13 22:42:36.194000+03:00


In [13]:
# Define your filter condition
condition = df_userInputData["item-used"] == 'Φαρμακευτικό αλκοόλ 95%'
# Get the indexes where the condition is True
matching_idx = df_userInputData.index[condition]
# Also get the next index (i+1), and make sure it's within bounds
next_idx = matching_idx + 1
next_idx = next_idx[(next_idx < len(df_userInputData)) & (next_idx >= 0)]

# Combine the matching indexes with the next ones
final_idx = matching_idx.union(next_idx)

# Filter the DataFrame
df_userInputDataExperiments = df_userInputData.loc[final_idx]
df_userInputDataExperiments


,_id,experimentState,timestamp,userInputCategory,are-doors-opened,are-people-inside,are-windows-opened,front-wall,item-used,notes,pollutant-type,quantity-used,room,side-right-wall,epoch_ms,timestamp_local
19,{'$oid': '684afcdf8ad2cd52978f638d'},InsertingSourcePollutant,{'$date': 1749744863788},ExperimentState,on,on,on,1.5,Φαρμακευτικό αλκοόλ 95%,,VOC,μία κουταλιά της σούπας,Σαλόνι,1.5,1749744863788,2025-06-12 19:14:23.788000+03:00
20,{'$oid': '684b0c908ad2cd52978f8641'},RemovingSourcePollutant,{'$date': 1749748880682},ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1749748880682,2025-06-12 20:21:20.682000+03:00
21,{'$oid': '684b12168ad2cd52978f92ea'},InsertingSourcePollutant,{'$date': 1749750294045},ExperimentState,on,on,on,1.5,Φαρμακευτικό αλκοόλ 95%,,VOC,13 ml περιπου,σαλονι,1.5,1749750294045,2025-06-12 20:44:54.045000+03:00
22,{'$oid': '684b1be48ad2cd52978fc054'},RemovingSourcePollutant,{'$date': 1749752804880},ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1749752804880,2025-06-12 21:26:44.880000+03:00
23,{'$oid': '684b21cb8ad2cd52978fdeca'},InsertingSourcePollutant,{'$date': 1749754315351},ExperimentState,on,on,on,1.5,Φαρμακευτικό αλκοόλ 95%,,VOC,17 ml,Σαλόνι,1.7,1749754315351,2025-06-12 21:51:55.351000+03:00
24,{'$oid': '684b2af0c4ba872d6a031472'},RemovingSourcePollutant,{'$date': 1749756656094},ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1749756656094,2025-06-12 22:30:56.094000+03:00
25,{'$oid': '684b3582440322b264da457e'},InsertingSourcePollutant,{'$date': 1749759362051},ExperimentState,on,on,on,1.5,Φαρμακευτικό αλκοόλ 95%,,VOC,10 ml,Σαλόνι,1.7,1749759362051,2025-06-12 23:16:02.051000+03:00
26,{'$oid': '684b3ebc99afe3f757053c92'},RemovingSourcePollutant,{'$date': 1749761724467},ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1749761724467,2025-06-12 23:55:24.467000+03:00
27,{'$oid': '684b4244c3e31ce70438ae27'},InsertingSourcePollutant,{'$date': 1749762628817},ExperimentState,on,on,on,1.5,Φαρμακευτικό αλκοόλ 95%,,VOC,13,Σαλόνι,1.7,1749762628817,2025-06-13 00:10:28.817000+03:00
28,{'$oid': '684b4b79b6e12fa45685e02c'},RemovingSourcePollutant,{'$date': 1749764985584},ExperimentState,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1749764985584,2025-06-13 00:49:45.584000+03:00


In [14]:
first_idx = df_userInputData.index[0]
last_idx = df_userInputData.index[-1]
start_time = df_userInputData.loc[first_idx,"epoch_ms"]
end_time   = df_userInputData.loc[last_idx,"epoch_ms"] 
first_part_endpoint = "timeSeriesEndpoints/"
second_part_endpoint = "postTimeSeriesData?start=" + str(start_time) + '&end=' +str(end_time)
file_name="TimeSeries"
create_url(base_url,first_part_endpoint,second_part_endpoint)
saveDataIntoDataFolder(url,file_name)
df_timeSeries = loadDataFromFile(file_name)

Received 83 records.
Data saved to /home/bob/Documents/Diploma project code/Diploma-Project/data/TimeSeries.json
File /home/bob/Documents/Diploma project code/Diploma-Project/dataAnalysis and machine learning/data/TimeSeries.json does not exist.


In [15]:
# 1. Select only the needed columns
df_timeSeriesDuringExperiments_copy = df_timeSeries[['timestamp', 'Id', 'BME680:breathVocEquivalent', 'CCS811:TVOC']].copy()

# 2. Convert `timestamp` (UTC) to Europe/Athens tz-aware datetime
df_timeSeriesDuringExperiments_copy['timestamp'] = (
    pd.to_datetime(df_timeSeriesDuringExperiments_copy['timestamp'], utc=True)
      .dt.tz_convert('Europe/Athens')
)
# 3. Create the three “Id=…:” columns
df_timeSeriesDuringExperiments_copy['Id=0:CCS811:TVOC'] = np.where(
    df_timeSeriesDuringExperiments_copy['Id'] == 0, df_timeSeriesDuringExperiments_copy['CCS811:TVOC'], np.nan
)
df_timeSeriesDuringExperiments_copy['Id=1:BME680:breathVocEquivalent'] = np.where(
    df_timeSeriesDuringExperiments_copy['Id'] == 1, df_timeSeriesDuringExperiments_copy['BME680:breathVocEquivalent'], np.nan
)
df_timeSeriesDuringExperiments_copy['Id=2:BME680:breathVocEquivalent'] = np.where(
    df_timeSeriesDuringExperiments_copy['Id'] == 2, df_timeSeriesDuringExperiments_copy['BME680:breathVocEquivalent'], np.nan
)

# 4. Set timestamp as the row index
df_timeSeriesDuringExperiments_copy.set_index('timestamp', inplace=True)

# 5. Drop the old Id+raw measurement columns
df_timeSeriesDuringExperiments_final = df_timeSeriesDuringExperiments_copy.drop(
    columns=['Id', 'BME680:breathVocEquivalent', 'CCS811:TVOC']
)

# Inspect result
df_timeSeriesDuringExperiments_final.head()
df_timeSeriesDuringExperiments_final

TypeError: 'NoneType' object is not subscriptable

In [ ]:
df_timeSeriesDuringExperiments_final

In [ ]:
# 1. Filter only rows 18 to 30 from df_userInputData
df_events = df_userInputData.iloc[19:30].reset_index(drop=True)
df_events


In [ ]:
# 2. Identify experiment periods based on Inserting/RemovingSourcePollutant
experiment_periods = []
for i in range(0, len(df_events) - 1, 2):
    start_row = df_events.iloc[i]
    end_row = df_events.iloc[i + 1]
    
    if (start_row['experimentState'] == 'InsertingSourcePollutant' and
        end_row['experimentState'] == 'RemovingSourcePollutant'):

        start_time = pd.to_datetime(start_row['timestamp_local'])
        end_time = pd.to_datetime(end_row['timestamp_local'])
        experiment_periods.append((start_time, end_time))

# 3. Create plots using Seaborn
num_experiments = len(experiment_periods)
print (experiment_periods)


In [ ]:
# Ensure the index is sorted and timezone-aware
df_timeSeriesDuringExperiments_final = df_timeSeriesDuringExperiments_final.sort_index()

# Find the actual bounds in the timeseries data for the experiment window
actual_start_time = df_timeSeriesDuringExperiments_final.index[
    df_timeSeriesDuringExperiments_final.index.searchsorted(start_time, side="left")
]

actual_end_time = df_timeSeriesDuringExperiments_final.index[
    df_timeSeriesDuringExperiments_final.index.searchsorted(end_time, side="right") - 1
]

# Slice the time series data between the actual bounds
df_period_data = df_timeSeriesDuringExperiments_final.loc[actual_start_time:actual_end_time]

In [ ]:
fig, axes = plt.subplots(num_experiments, 2, figsize=(14, 4 * num_experiments), sharex=False)

# Ensure axes is always a 2D array
if num_experiments == 1:
    axes = np.array([axes])

for idx, (start, end) in enumerate(experiment_periods):
    # Slice time series data between start and end
    df_range = df_timeSeriesDuringExperiments_final.loc[start:end]

    # LEFT PLOT: Id=0:CCS811:TVOC
    sns.lineplot(
        ax=axes[idx, 0],
        data=df_range,
        x=df_range.index,
        y='Id=0:CCS811:TVOC',
        marker='o'
    )
    axes[idx, 0].set_title(f"data between {start} and {end}\nId=0:CCS811:TVOC")
    axes[idx, 0].set_xlabel("Timestamp")
    axes[idx, 0].set_ylabel("TVOC")

    # RIGHT PLOT: Id=1 and Id=2 for BME680
    sns.lineplot(
        ax=axes[idx, 1],
        data=df_range,
        x=df_range.index,
        y='Id=1:BME680:breathVocEquivalent',
        label="Id=1",
        marker='o'
    )
    sns.lineplot(
        ax=axes[idx, 1],
        data=df_range,
        x=df_range.index,
        y='Id=2:BME680:breathVocEquivalent',
        label="Id=2",
        marker='s'
    )
    axes[idx, 1].set_title(f"data between {start} and {end}\nBME680:breathVocEquivalent")
    axes[idx, 1].set_xlabel("Timestamp")
    axes[idx, 1].set_ylabel("VOC")
    axes[idx, 1].legend()

# Adjust layout
plt.tight_layout()
plt.show()